# Solid Principles

*Run each cell with **Shift+Enter***

02 — SOLID Principles, One Focused Before/After Each
===================================================

Runnable companion to PDF Book IV "The five principles that make OO code change-friendly".

SOLID is not dogma — it's five levers for LOW COUPLING and HIGH COHESION so that
change stays cheap. This file demonstrates each principle with the smallest
honest "smell → fix" pair, and asserts the fixed design actually behaves:

    S  Single Responsibility  — one reason to change per class
    O  Open/Closed            — extend without editing tested code
    L  Liskov Substitution    — subtypes must honor the base type's contract
    I  Interface Segregation  — many small roles beat one fat interface
    D  Dependency Inversion   — depend on abstractions, inject concretions

Run:  python solid_principles.py

In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Protocol

===========================================================================
S — SINGLE RESPONSIBILITY: split "compute" from "format" from "persist"
===========================================================================
SMELL: a Report class that calculates totals, renders HTML, AND saves files
changes for three unrelated reasons. Split it so each class has ONE job.

In [ ]:
class SalesReport:
    def __init__(self, sales: list[float]):
        self.sales = sales

    def total(self) -> float:                 # the ONLY reason this class changes
        return sum(self.sales)


class HtmlReportFormatter:
    def render(self, report: SalesReport) -> str:   # formatting is a separate axis
        return f"<h1>Total: {report.total():.2f}</h1>"


class ReportRepository:
    def __init__(self):
        self.saved: dict[str, str] = {}

    def save(self, name: str, content: str) -> None:  # persistence is a third axis
        self.saved[name] = content

===========================================================================
O — OPEN/CLOSED: add behavior by adding a class, not editing an if-ladder
===========================================================================
SMELL: `if shape == "circle": ... elif "square": ...` — every new shape edits
a tested function. FIX: a polymorphic type; new shapes are NEW classes.

In [ ]:
class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...


class Circle(Shape):
    def __init__(self, r: float): self.r = r
    def area(self) -> float: return 3.14159 * self.r * self.r


class Square(Shape):
    def __init__(self, s: float): self.s = s
    def area(self) -> float: return self.s * self.s


class Triangle(Shape):                         # added WITHOUT touching total_area
    def __init__(self, b: float, h: float): self.b, self.h = b, h
    def area(self) -> float: return 0.5 * self.b * self.h


def total_area(shapes: list[Shape]) -> float:  # closed for modification
    return sum(s.area() for s in shapes)       # open for extension (new Shape)

===========================================================================
L — LISKOV SUBSTITUTION: a subtype must be usable wherever the base is
===========================================================================
SMELL: the classic Square(Rectangle) — overriding width to also set height
breaks code that trusts Rectangle's contract. FIX: model them as siblings so
no subtype weakens a base guarantee.

In [ ]:
class Rectangle:
    def __init__(self, w: float, h: float): self._w, self._h = w, h
    def area(self) -> float: return self._w * self._h


class SquareLSP:
    def __init__(self, side: float): self._s = side
    def area(self) -> float: return self._s * self._s


def area_of(shape) -> float:                   # works for BOTH; neither surprises
    return shape.area()

===========================================================================
I — INTERFACE SEGREGATION: don't force clients to implement methods they skip
===========================================================================
SMELL: one Worker interface with work() AND eat() forces a RobotWorker to
implement a meaningless eat(). FIX: split into focused roles (Protocols).

In [ ]:
class Workable(Protocol):
    def work(self) -> str: ...


class Eatable(Protocol):
    def eat(self) -> str: ...


class Human:                                   # implements BOTH roles
    def work(self) -> str: return "coding"
    def eat(self) -> str: return "lunch"


class Robot:                                   # implements ONLY what it needs
    def work(self) -> str: return "welding"


def run_shift(workers: list[Workable]) -> list[str]:
    return [w.work() for w in workers]         # asks only for Workable

===========================================================================
D — DEPENDENCY INVERSION: high-level policy depends on an abstraction
===========================================================================
SMELL: OrderService constructs a concrete MySqlDatabase inside itself — you
can't test it without a real DB and can't swap stores. FIX: depend on a
NotificationGateway abstraction and INJECT the concrete one.

In [ ]:
class NotificationGateway(Protocol):
    def send(self, to: str, msg: str) -> None: ...


class EmailGateway:
    def __init__(self): self.sent: list[tuple[str, str]] = []
    def send(self, to: str, msg: str) -> None: self.sent.append((to, msg))


class SmsGateway:
    def __init__(self): self.sent: list[tuple[str, str]] = []
    def send(self, to: str, msg: str) -> None: self.sent.append((to, msg))


class OrderService:
    def __init__(self, gateway: NotificationGateway):   # inject the abstraction
        self._gateway = gateway

    def place(self, customer: str) -> None:
        self._gateway.send(customer, "Order confirmed")  # no idea which concrete


def demo() -> None:
    # S — each class does one job and composes cleanly.
    report = SalesReport([100.0, 250.0, 50.0])
    html = HtmlReportFormatter().render(report)
    repo = ReportRepository()
    repo.save("q1", html)
    assert report.total() == 400.0
    assert "400.00" in html and repo.saved["q1"] == html
    print("   S · SalesReport / Formatter / Repository each change for one reason")

    # O — add Triangle without editing total_area.
    shapes: list[Shape] = [Circle(1), Square(2), Triangle(3, 4)]
    assert round(total_area(shapes), 2) == round(3.14159 + 4 + 6, 2)
    print("   O · total_area unchanged; Triangle added as a new class")

    # L — both shapes are substitutable in area_of with no surprises.
    assert area_of(Rectangle(3, 4)) == 12
    assert area_of(SquareLSP(5)) == 25
    print("   L · Rectangle and SquareLSP each honor the area() contract")

    # I — Robot needn't implement eat(); run_shift only needs Workable.
    assert run_shift([Human(), Robot()]) == ["coding", "welding"]
    print("   I · Robot implements only work(); no dead eat() method forced")

    # D — same service, swap the injected gateway with zero changes.
    email = EmailGateway()
    OrderService(email).place("ada@example.com")
    sms = SmsGateway()
    OrderService(sms).place("+15551234")
    assert email.sent == [("ada@example.com", "Order confirmed")]
    assert sms.sent == [("+15551234", "Order confirmed")]
    print("   D · OrderService depends on the gateway abstraction; concretion injected")


def main() -> None:
    print("=" * 70)
    print("SOLID — solid_principles.py")
    print("=" * 70)
    print("Five levers for low coupling / high cohesion (smell -> fix):")
    demo()
    print("-" * 70)
    print("Lesson: SOLID exists to keep CHANGE cheap — apply it where change actually happens, not everywhere.")
    print("All solid_principles demos passed ✔")


if __name__ == "__main__":
    # Keep Unicode output safe even when stdout is redirected/piped (Windows cp1252 fallback).
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()

---
# SOLID — Exhaustive Notebook


# Solid Notebook

*Run each cell with **Shift+Enter***

SOLID Principles — Exhaustive Notebook
=======================================
Five levers for LOW COUPLING and HIGH COHESION.

SOLID is not a checklist — it is a diagnostic tool.
Each principle answers: "When this requirement changes,
how much code MUST change with it?"

PART S · Single Responsibility Principle
PART O · Open / Closed Principle
PART L · Liskov Substitution Principle
PART I · Interface Segregation Principle
PART D · Dependency Inversion Principle
PART X · How the principles interlock
PART Y · When NOT to apply SOLID (the YAGNI trap)

Run: python solid_notebook.py

In [ ]:
from __future__ import annotations

import sys
from abc import ABC, abstractmethod
from typing import Protocol


def sep(t: str) -> None:
    print(f"\n{'═'*64}\n  {t}\n{'═'*64}")

## PART S — SINGLE RESPONSIBILITY PRINCIPLE

Definition: A class should have ONE reason to change.

Mental model: "Who orders this class to change?"
  If the answer is "the business analytics team AND the ops team
  AND the UI team", you have three responsibilities in one class.
  Split it so each team owns exactly one class.

GOTCHA 1: SRP is about ACTORS (who orders change), NOT about method count.
  A class with 20 methods that all serve the same actor is fine.
  A class with 2 methods serving different actors is a violation.

GOTCHA 2: Over-applying SRP fragments code into dozens of one-liners
  (the "Ravioli code" anti-pattern). Apply it when you feel actual pain —
  when two unrelated changes collide in the same class, not preemptively.

GOTCHA 3: "Cohesion" is the positive framing of SRP.
  High cohesion = all parts of a class serve the same purpose.
  Low cohesion = the class is a junk drawer.

In [ ]:
def notebook_srp() -> None:

## S · Single Responsibility Principle

In [ ]:
# ── SMELL: God class — three actors, one class ────────────────────────
    # This class changes when:
    #   1. The finance team changes how totals are calculated
    #   2. The design team changes how reports look
    #   3. The ops team changes where reports are stored
    class GodReport:
        def __init__(self, sales: list[float]):
            self.sales = sales

        def total(self) -> float:            # axis 1: finance logic
            return sum(self.sales)

        def to_html(self) -> str:            # axis 2: presentation
            return f"<h1>Total: {self.total():.2f}</h1>"

        def save(self, path: str) -> None:   # axis 3: persistence
            # In real code: open(path, "w").write(self.to_html())
            pass

    # ── FIX: each class serves one actor ─────────────────────────────────
    class SalesReport:
        def __init__(self, sales: list[float]): self.sales = sales
        def total(self) -> float: return sum(self.sales)  # ONLY finance actor

    class HtmlFormatter:
        def render(self, r: SalesReport) -> str:          # ONLY design actor
            return f"<h1>Total: {r.total():.2f}</h1>"

    class FileRepository:
        def __init__(self): self._store: dict[str, str] = {}
        def save(self, name: str, content: str) -> None:  # ONLY ops actor
            self._store[name] = content
        def load(self, name: str) -> str:
            return self._store[name]

    r = SalesReport([100, 250, 50])
    html = HtmlFormatter().render(r)
    repo = FileRepository()
    repo.save("q1", html)
    assert r.total() == 400.0
    assert "400.00" in html
    print(f"SRP: report={r.total()}, html={html[:30]!r}")

    # ── GOTCHA 1: SRP is about actors, not method count ────────────────────
    # This class has many methods — but ONE actor (finance) orders all changes.
    # It does NOT violate SRP.
    class TaxCalculator:
        TAX_RATE = 0.08
        def gross(self, amount: float) -> float:     return amount
        def tax(self,   amount: float) -> float:     return amount * self.TAX_RATE
        def net(self,   amount: float) -> float:     return amount + self.tax(amount)
        def rounded(self, amount: float) -> float:   return round(self.net(amount), 2)
        def display(self, amount: float) -> str:     return f"${self.rounded(amount)}"

    tc = TaxCalculator()
    assert tc.rounded(100) == 108.0
    print(f"TaxCalculator (many methods, ONE actor): {tc.display(100)}")
    print("  → does NOT violate SRP — finance team owns all these methods")

    # ── GOTCHA 2: recognising SRP violations by change collision ───────────
    # If two unrelated bugs are in the same file simultaneously,
    # and fixing one risks breaking the other → SRP violation!
    print("\n  Smell to watch for: two open PRs touching the same class")
    print("  for unrelated reasons. Split the class.")

## PART O — OPEN / CLOSED PRINCIPLE

Definition: Software entities should be OPEN for extension,
            CLOSED for modification.

Mental model: Adding a new feature should add new code, not edit
  existing, working code. You extend via new classes/functions,
  not by modifying tested ones.

GOTCHA 1: The if/elif type ladder is the canonical OCP smell.
  Every new type requires editing a tested function.
GOTCHA 2: OCP does NOT mean "never touch old code."
  Fixing a bug in existing code is fine. OCP is about EXTENSION.
GOTCHA 3: OCP requires an ABSTRACTION to extend against.
  Without a stable base abstraction, you can't add without editing.
GOTCHA 4: Over-engineering OCP creates unnecessary abstractions.
  Apply it when you have (or foresee) a SECOND implementation.

In [ ]:
def notebook_ocp() -> None:

## O · Open / Closed Principle

In [ ]:
# ── SMELL: if/elif ladder — adding Hexagon edits tested code ──────────
    def total_area_bad(shapes: list[dict]) -> float:
        total = 0.0
        for s in shapes:
            if s["type"] == "circle":       # every new shape: edit here
                total += 3.14159 * s["r"]**2
            elif s["type"] == "square":
                total += s["side"]**2
            # elif "hexagon": ...   ← must edit tested code
        return total

    # ── FIX: polymorphic type hierarchy — new shapes add new classes ───────
    class Shape(ABC):
        @abstractmethod
        def area(self) -> float: ...

    class Circle(Shape):
        def __init__(self, r: float): self.r = r
        def area(self) -> float: return 3.14159 * self.r**2

    class Square(Shape):
        def __init__(self, s: float): self.s = s
        def area(self) -> float: return self.s**2

    class Triangle(Shape):                        # ADDED — zero edits to total_area
        def __init__(self, b: float, h: float): self.b, self.h = b, h
        def area(self) -> float: return 0.5 * self.b * self.h

    class Hexagon(Shape):                         # ADDED AGAIN — still zero edits
        def __init__(self, side: float): self.side = side
        def area(self) -> float: return (3 * (3**0.5) / 2) * self.side**2

    def total_area(shapes: list[Shape]) -> float:  # CLOSED for modification
        return sum(s.area() for s in shapes)       # OPEN for extension

    shapes: list[Shape] = [Circle(1), Square(2), Triangle(3,4), Hexagon(1)]
    print(f"total_area with 4 shape types: {total_area(shapes):.4f}")
    print("  → added Triangle and Hexagon WITHOUT touching total_area ✓")

    # ── OCP with strategy pattern (functional style) ───────────────────────
    # OCP isn't only about class hierarchies — functions and strategy objects work too.
    from typing import Callable

    Pricer = Callable[[float], float]

    def standard(amount: float) -> float: return amount
    def premium(amount: float):           return amount * 0.9   # 10% discount
    def vip(amount: float):               return amount * 0.8   # 20% discount
    # Adding a new pricing strategy = adding a new function, editing nothing.

    def checkout(amount: float, pricer: Pricer) -> float:   # CLOSED
        return round(pricer(amount), 2)

    assert checkout(100, standard) == 100.00
    assert checkout(100, premium)  == 90.00
    assert checkout(100, vip)      == 80.00
    print(f"\nStrategy OCP: standard={checkout(100,standard)}, premium={checkout(100,premium)}")

    # ── GOTCHA 3: need a STABLE abstraction to extend against ──────────────
    # If the base Shape interface changes (e.g. adding perimeter()),
    # ALL subclasses must change — the "abstraction" was unstable.
    # Lesson: design the abstraction carefully before you start extending it.
    print("\n  GOTCHA: unstable base interface forces changes everywhere.")
    print("  Design the abstraction BEFORE building multiple implementations.")

    # ── GOTCHA 4: don't abstract prematurely ──────────────────────────────
    # With only ONE implementation, an abstraction adds indirection for no gain.
    # Apply OCP when the SECOND implementation arrives, not before.
    print("  YAGNI: don't create an abstract Serialiser with ONE JsonSerializer.")
    print("  Refactor when the second implementation is needed.")

## PART L — LISKOV SUBSTITUTION PRINCIPLE

Definition: Objects of a subtype must be substitutable for objects
  of their parent type without changing the correctness of the program.

Formal rules (Barbara Liskov, 1987):
  • Preconditions cannot be STRENGTHENED in a subtype.
  • Postconditions cannot be WEAKENED in a subtype.
  • Invariants of the supertype must hold in the subtype.
  • History constraint: subtype must not allow state changes that
    the supertype would not allow.

Mental model: If you have a function that works with a Rectangle,
  it must work identically with any subclass of Rectangle.
  "Works identically" = produces the same guarantees, not the same values.

GOTCHA 1: Square(Rectangle) is the CLASSIC LSP violation.
GOTCHA 2: Raising NotImplementedError in overridden methods violates LSP.
GOTCHA 3: isinstance() checks in callers signal LSP violations.
GOTCHA 4: Widening exceptions in overrides violates LSP.
GOTCHA 5: LSP applies to PROTOCOLS too, not just class inheritance.

In [ ]:
def notebook_lsp() -> None:

## L · Liskov Substitution Principle

In [ ]:
# ── GOTCHA 1: Square(Rectangle) — the canonical violation ─────────────
    # Rectangle contract: width and height are INDEPENDENT.
    # Setting width to 5 keeps height at its current value.
    class Rectangle:
        def __init__(self, w: float, h: float): self._w, self._h = w, h
        @property
        def width(self)  -> float: return self._w
        @property
        def height(self) -> float: return self._h
        @width.setter
        def width(self, v: float):  self._w = v    # only width changes
        @height.setter
        def height(self, v: float): self._h = v    # only height changes
        def area(self) -> float: return self._w * self._h

    class BadSquare(Rectangle):
        """LSP VIOLATION: overrides setters to enforce equal sides,
           BREAKING the Rectangle contract that width/height are independent."""
        @Rectangle.width.setter
        def width(self, v: float):
            self._w = self._h = v   # setting width ALSO changes height!
        @Rectangle.height.setter
        def height(self, v: float):
            self._w = self._h = v   # setting height ALSO changes width!

    def resize_and_measure(rect: Rectangle) -> float:
        """Expects Rectangle contract: set width, height stays the same."""
        rect.width = 5
        rect.height = 4
        return rect.area()

    r = Rectangle(3, 3)
    sq = BadSquare(3, 3)
    area_r  = resize_and_measure(r)
    area_sq = resize_and_measure(sq)
    print(f"Rectangle area after resize: {area_r}")    # 20 (5*4 = correct)
    print(f"BadSquare  area after resize: {area_sq}")  # 16 (4*4 — LSP VIOLATION!)
    assert area_r == 20 and area_sq != 20
    print("  → BadSquare BREAKS code that expects Rectangle's contract!")

    # ── FIX: don't inherit — make them siblings under a common Shape ───────
    class GoodSquare:
        def __init__(self, side: float): self._s = side
        def area(self) -> float: return self._s ** 2

    # Now area_of() can work with either, but they make different guarantees:
    def area_of(shape) -> float: return shape.area()

    assert area_of(Rectangle(5, 4)) == 20
    assert area_of(GoodSquare(5))   == 25
    print("FIX: Square is NOT a Rectangle — they're siblings under Shape ✓")

    # ── GOTCHA 2: NotImplementedError stub violates LSP ────────────────────
    class ReadWriteStorage(ABC):
        @abstractmethod
        def read(self, key: str) -> str: ...
        @abstractmethod
        def write(self, key: str, val: str) -> None: ...

    class ReadOnlyStorage(ReadWriteStorage):
        def __init__(self, data: dict): self._d = data
        def read(self, key: str) -> str: return self._d[key]
        def write(self, key: str, val: str) -> None:
            raise NotImplementedError("read-only!")  # VIOLATES LSP!
            # Code expecting ReadWriteStorage will explode at runtime.

    # FIX: segregate the interface (ISP) — don't inherit if you can't fulfil
    class Readable(Protocol):
        def read(self, key: str) -> str: ...
    class Writable(Protocol):
        def write(self, key: str, val: str) -> None: ...

    class TrueReadOnly:
        def __init__(self, data: dict): self._d = data
        def read(self, key: str) -> str: return self._d[key]
        # No write() — not even in the type. Callers of Readable cannot call write().

    tro = TrueReadOnly({"x": "hello"})
    assert tro.read("x") == "hello"
    print("\nFIX: TrueReadOnly implements ONLY Readable — no stub write() ✓")

    # ── GOTCHA 3: isinstance() checks are a LSP smell ─────────────────────
    def process_bad(payment) -> str:
        if isinstance(payment, type(payment)):  # simplified — real code: isinstance(p, CryptoPayment)
            pass  # special-case handling = tells you LSP is violated
        return "processed"

    print("\n  SMELL: isinstance(x, SubType) in business logic = LSP violation.")
    print("  The caller shouldn't need to know WHICH subtype it has.")
    print("  Fix: push the special behaviour into the subtype via override.")

    # ── GOTCHA 4: strengthened preconditions violate LSP ──────────────────
    class BaseValidator:
        def validate(self, value: int) -> bool:
            return value >= 0       # accepts any non-negative

    class StrictValidator(BaseValidator):
        def validate(self, value: int) -> bool:
            return value >= 100     # STRENGTHENED: rejects 0..99 — violates LSP!

    base   = BaseValidator()
    strict = StrictValidator()
    print(f"\nBase accepts 50: {base.validate(50)}")     # True
    print(f"Strict accepts 50: {strict.validate(50)}")   # False — LSP violation!
    print("  RULE: subtype preconditions must be EQUAL OR WEAKER than parent.")

## PART I — INTERFACE SEGREGATION PRINCIPLE

Definition: Clients should not be forced to depend on methods
  they do not use.

Mental model: Many small, focused interfaces > one fat interface.
  A "fat interface" forces every implementer to provide ALL methods
  even when they only need a subset. The unused methods end up as
  stubs that throw — which is an LSP violation waiting to happen.

GOTCHA 1: Fat ABCs with abstract methods force NotImplementedError stubs.
GOTCHA 2: In Python, Protocol is the preferred ISP mechanism.
  Structural typing: no explicit inheritance required.
GOTCHA 3: ISP != "one method per interface". Group methods that change
  together for the SAME reason. An interface is cohesive, not minimal.
GOTCHA 4: Too many tiny interfaces cause "interface soup" — as harmful
  as too few. Find the natural role boundaries.

In [ ]:
def notebook_isp() -> None:

## I · Interface Segregation Principle

In [ ]:
# ── SMELL: fat interface forces Robot to stub eat() ────────────────────
    class FatWorker(ABC):
        @abstractmethod
        def work(self) -> str: ...
        @abstractmethod
        def eat(self) -> str: ...           # Robot can't eat!
        @abstractmethod
        def sleep(self) -> str: ...         # Robot can't sleep!

    class HumanWorker(FatWorker):
        def work(self)  -> str: return "coding"
        def eat(self)   -> str: return "eating"
        def sleep(self) -> str: return "sleeping"

    class RobotWorker(FatWorker):
        def work(self) -> str: return "welding"
        def eat(self)  -> str: raise NotImplementedError("robots don't eat!")  # FORCED stub
        def sleep(self)-> str: raise NotImplementedError("robots don't sleep!")

    robot = RobotWorker()
    try:
        robot.eat()
    except NotImplementedError as e:
        print(f"Fat interface forces bad stub: {e}")

    # ── FIX: segregated Protocol roles ────────────────────────────────────
    class Workable(Protocol):
        def work(self) -> str: ...

    class Eatable(Protocol):
        def eat(self) -> str: ...

    class Resting(Protocol):
        def sleep(self) -> str: ...

    class Human:                      # implements all three roles naturally
        def work(self)  -> str: return "coding"
        def eat(self)   -> str: return "eating"
        def sleep(self) -> str: return "sleeping"

    class Robot:                      # implements ONLY what it does
        def work(self)  -> str: return "welding"
        # No eat() or sleep() — not in its type at all!

    def run_shift(workers: list[Workable]) -> list[str]:
        return [w.work() for w in workers]  # asks only for Workable

    def lunch_break(eaters: list[Eatable]) -> list[str]:
        return [e.eat() for e in eaters]   # Robot can't be passed here → type error!

    result = run_shift([Human(), Robot()])   # Robot IS Workable ✓
    assert result == ["coding", "welding"]
    print(f"\nrun_shift with Human+Robot: {result}")

    lunchees = lunch_break([Human()])        # Robot excluded by type ✓
    assert lunchees == ["eating"]
    print(f"lunch_break (Human only, Robot excluded): {lunchees}")

    # ── GOTCHA 3: ISP isn't about "one method per interface" ──────────────
    # read() and write() change for the SAME reason (persistence contract).
    # They belong together even though a read-only store might split them.
    class DataStore(Protocol):
        def read(self, key: str) -> str: ...
        def write(self, key: str, val: str) -> None: ...
        def delete(self, key: str) -> None: ...

    # But split when roles diverge:
    class Reader(Protocol):
        def read(self, key: str) -> str: ...

    class Writer(Protocol):
        def write(self, key: str, val: str) -> None: ...
        def delete(self, key: str) -> None: ...

    print("\n  RULE: group methods by role (who changes them), not just by count.")
    print("  read+write+delete go together; worker+eater are separate roles.")

    # ── GOTCHA 4: ISP and dependency size ─────────────────────────────────
    # A caller that only reads should depend on Reader, not DataStore.
    # This way a change to write() or delete() doesn't require recompiling
    # (or retesting) the read-only caller.
    print("\n  BENEFIT: narrow dependency → isolated recompilation / retesting.")
    print("  A read-only service depending on Reader won't break when write() changes.")

## PART D — DEPENDENCY INVERSION PRINCIPLE

Definition:
  A. High-level modules should not depend on low-level modules.
     Both should depend on ABSTRACTIONS.
  B. Abstractions should not depend on details.
     Details should depend on abstractions.

Mental model: The high-level policy (business logic) should not
  need to change when the low-level detail (database, email provider,
  payment processor) changes. Flip the dependency arrow: instead of
  OrderService importing MySqlDB, MySqlDB implements the StoragePort
  that OrderService depends on.

KEY DISTINCTION: Dependency Injection (DI) is the MECHANISM.
  Dependency Inversion (DIP) is the PRINCIPLE.
  You can inject a concrete class (DI without DIP — the dep still points wrong).
  DIP = inject an ABSTRACTION; the concrete is decided at the COMPOSITION ROOT.

GOTCHA 1: "new is glue" — constructing a dependency inside a class
  couples it forever; you can never swap it or test in isolation.
GOTCHA 2: Injecting a concrete type still couples to it (DI without DIP).
GOTCHA 3: The composition root (where abstractions bind to concretions)
  belongs at the EDGE of your system, not deep in business logic.
GOTCHA 4: Over-abstracting for DIP creates a one-implementation Protocol
  for every class — speculative generality with no payoff.

In [ ]:
def notebook_dip() -> None:

## D · Dependency Inversion Principle

In [ ]:
# ── SMELL 1: "new is glue" — constructing inside the class ────────────
    class BadOrderService:
        def __init__(self):
            self._db = {}          # concrete dict DB — hardcoded!
            # You can NEVER inject a real DB, a test stub, or Redis here.

        def place(self, customer: str, amount: float) -> None:
            self._db[customer] = amount   # coupled to in-memory dict forever

    bad = BadOrderService()
    bad.place("alice", 99.0)
    print("BadOrderService: works, but untestable and unswappable")

    # ── SMELL 2: injecting concrete instead of abstract ───────────────────
    class ConcreteDb:
        def __init__(self): self.data: dict = {}
        def save(self, k: str, v: float) -> None: self.data[k] = v

    class StillCoupledService:
        def __init__(self, db: ConcreteDb):    # DI without DIP — still coupled!
            self._db = db

    # ^ If ConcreteDb changes signature or you need a different store,
    # you MUST change StillCoupledService — the coupling is still wrong.

    # ── FIX: invert the dependency through an abstraction ─────────────────
    class StoragePort(Protocol):
        def save(self, customer: str, amount: float) -> None: ...

    class ChargePort(Protocol):
        def charge(self, customer: str, amount: float) -> str: ...

    class NotifyPort(Protocol):
        def notify(self, customer: str, ref: str) -> None: ...

    # High-level policy — depends on abstractions, knows nothing about concrete
    class OrderService:
        def __init__(self, storage: StoragePort,
                           charge:  ChargePort,
                           notify:  NotifyPort) -> None:
            self._storage = storage
            self._charge  = charge
            self._notify  = notify

        def place(self, customer: str, amount: float) -> str:
            ref = self._charge.charge(customer, amount)
            self._storage.save(customer, amount)
            self._notify.notify(customer, ref)
            return ref

    # Low-level details — implement the ports
    class InMemoryStorage:
        def __init__(self): self.records: dict = {}
        def save(self, c: str, a: float) -> None: self.records[c] = a

    class FakeStripe:
        def __init__(self): self.charges: list = []
        def charge(self, c: str, a: float) -> str:
            self.charges.append((c, a))
            return f"stripe_{c}"

    class SpyNotifier:
        def __init__(self): self.sent: list = []
        def notify(self, c: str, ref: str) -> None: self.sent.append((c,ref))

    # ── COMPOSITION ROOT: bind abstractions to concretions ────────────────
    storage  = InMemoryStorage()
    stripe   = FakeStripe()
    notifier = SpyNotifier()
    svc = OrderService(storage, stripe, notifier)   # composition root

    ref = svc.place("alice", 99.0)
    assert ref == "stripe_alice"
    assert storage.records["alice"] == 99.0
    assert notifier.sent[0] == ("alice", "stripe_alice")
    print(f"\nDIP: OrderService placed order, ref={ref!r}")

    # Swap to a different payment provider — ZERO changes to OrderService
    class FakePayPal:
        def charge(self, c: str, a: float) -> str: return f"paypal_{c}"

    svc2 = OrderService(storage, FakePayPal(), notifier)
    ref2 = svc2.place("bob", 50.0)
    assert ref2 == "paypal_bob"
    print(f"Swapped to PayPal:            ref={ref2!r}  — OrderService unchanged ✓")

    # ── GOTCHA 1: "new is glue" summary ───────────────────────────────────
    print("\n  RULE: 'new is glue'. If a class constructs its own collaborators,")
    print("  it's glued to them. Inject collaborators to cut the glue.")

    # ── GOTCHA 3: where is the composition root? ──────────────────────────
    print("\n  COMPOSITION ROOT belongs at the EDGE of the system:")
    print("    FastAPI → main.py / Depends() wiring")
    print("    Tests   → the test setup (conftest.py / fixture)")
    print("    CLI     → main() function")
    print("  NOT inside domain classes, service layers, or utilities.")

    # ── DI vs DIP in one diagram ───────────────────────────────────────────
    print("""
  Dependency INJECTION (mechanism):
    def __init__(self, db):  ← db is injected from outside
    Problem if db is concrete: coupling still exists

  Dependency INVERSION (principle):
    def __init__(self, port: StoragePort):  ← depends on abstraction
    The arrow is INVERTED: port depends on the high-level policy, not vice versa
""")

## PART X — HOW THE PRINCIPLES INTERLOCK

The five principles are NOT independent — they REINFORCE each other.
Understanding the connections is a senior-level signal.

In [ ]:
def notebook_interlock() -> None:

## X · How SOLID Principles Interlock

In [ ]:
print("""
  ┌─────────────────────────────────────────────────────────────────┐
  │  SRP → ISP → DIP  (the cohesion chain)                        │
  │                                                                   │
  │  SRP splits classes so each has ONE reason to change.            │
  │  This naturally produces SMALL, cohesive interfaces (ISP).       │
  │  Small interfaces are easy to depend on abstractly (DIP).        │
  └─────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────┐
  │  ISP violation ──► LSP violation (forced path)                  │
  │                                                                   │
  │  A fat interface forces implementers to stub unused methods.     │
  │  Those stubs raise NotImplementedError.                          │
  │  Code that calls via the fat interface will explode at runtime.  │
  │  That explosion IS an LSP violation.                             │
  └─────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────┐
  │  OCP requires DIP (you need an abstraction to extend against)   │
  │                                                                   │
  │  OCP says "add classes, don't edit existing ones."              │
  │  But total_area() can't be closed without a Shape abstraction.  │
  │  The abstraction is the DIP contract.                            │
  │  Without DIP, there's nothing stable enough to be closed.       │
  └─────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────┐
  │  LSP is what MAKES OCP safe                                     │
  │                                                                   │
  │  OCP says "pass new shapes to total_area() without editing it." │
  │  This only works if new shapes satisfy the Shape contract (LSP). │
  │  An LSP-violating shape would silently produce wrong results.    │
  └─────────────────────────────────────────────────────────────────┘
""")

## PART Y — WHEN NOT TO APPLY SOLID (the YAGNI trap)

SOLID has a COST: every abstraction is a layer of indirection.
Over-applying SOLID produces "maze code" — dozens of one-class
interfaces that make simple things hard to follow.

The failure mode of a MID-LEVEL engineer is applying SOLID everywhere.
The failure mode of a JUNIOR engineer is not applying it at all.
The SENIOR level is knowing WHEN the pain justifies the abstraction.

In [ ]:
def notebook_when_not_to() -> None:

## Y · When NOT to Apply SOLID

In [ ]:
print("""
  RULE: introduce an abstraction when you have (or clearly foresee) a
  SECOND implementation, OR when you need a SEAM for testing.

  ┌─────────────────────────────────────────────────────────────────┐
  │  Over-engineering smell: "one-implementation interface"         │
  │                                                                   │
  │  class UserRepository(ABC):   ← only ever one subclass          │
  │      @abstractmethod          ← extra file, extra ceremony      │
  │      def find(self, id): ...  ← but you added it "for SOLID"   │
  │                                                                   │
  │  If there's only ONE concrete UserRepository, the abstraction   │
  │  adds complexity with no benefit. YAGNI.                         │
  └─────────────────────────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────────────────────────┐
  │  SRP over-applied: "Ravioli code"                               │
  │                                                                   │
  │  class ValidateEmail: ...                                        │
  │  class ValidateAge: ...                                          │
  │  class ValidatePhone: ...                                        │
  │  class ValidateAll: ...  ← orchestrates the above               │
  │                                                                   │
  │  Three one-liners extracted into three classes.                  │
  │  The "cohesion" argument doesn't hold when things are this tiny. │
  └─────────────────────────────────────────────────────────────────┘

  WHEN to abstract:
    ✓ You have TWO concrete implementations now.
    ✓ You need to inject a test double (fake/stub/spy).
    ✓ A change in the detail (e.g., DB schema) is repeatedly forcing
      changes in the policy (e.g., service layer).
    ✓ The team is editing the same class for unrelated reasons simultaneously.

  When NOT to abstract:
    ✗ You "might" need it in the future (speculative generality).
    ✗ One implementation, never tested in isolation.
    ✗ The abstraction only exists to satisfy a style guide.
""")

## PART Z — INTERVIEW QUESTIONS & GOTCHA CHECKLIST

In [ ]:
def notebook_interview() -> None:

## Z · Interview Questions & Gotcha Checklist

In [ ]:
print("""
  Q1: "What does SRP actually mean?" (Junior trap)
  ─────────────────────────────────────────────────
  Wrong: "One method per class" or "small classes"
  Right: "One REASON TO CHANGE" — one actor (business role) that
          orders the class to be modified. Measure by actors, not methods.

  Q2: "Give an example of an LSP violation" (Senior test)
  ──────────────────────────────────────────────────────
  Classic: Square(Rectangle) — overriding setters that the parent's
  contract promises are independent. Code expecting Rectangle breaks.
  Also: NotImplementedError stubs, narrowed preconditions.

  Q3: "What is the difference between DI and DIP?" (Principal-level)
  ──────────────────────────────────────────────────────────────────
  DI = Dependency Injection: passing collaborators from outside (mechanism).
  DIP = Dependency Inversion: depending on an ABSTRACTION, not a concretion
        (principle). You can have DI without DIP (inject a concrete class).
  DIP flips the arrow: the detail implements the port that the policy owns.

  Q4: "When would you NOT apply SOLID?" (Wisdom check)
  ──────────────────────────────────────────────────────
  When there's only one implementation, no need for test seams, and
  the abstraction adds indirection without any benefit (YAGNI).
  Introduce abstractions when the SECOND use case arrives, not speculatively.

  Q5: "How do ISP and LSP relate?" (Senior architecture)
  ──────────────────────────────────────────────────────
  ISP violation forces implementers to stub unused methods (NotImplementedError).
  Those stubs ARE LSP violations — callers expecting the full interface will fail.
  ISP violations create LSP violations downstream.

  GOTCHA CHECKLIST
  ─────────────────
  ✓ instanceof checks in business logic → LSP violation (subtype surprises callers)
  ✓ raise NotImplementedError in an override → ISP + LSP violation
  ✓ if/elif type ladder that grows with new types → OCP violation
  ✓ service class constructing its own DB / HTTP client → DIP violation ("new is glue")
  ✓ injecting a concrete class rather than an abstraction → DI but NOT DIP
  ✓ one-implementation abstract class → over-engineering (YAGNI)
  ✓ class that multiple unrelated teams edit → SRP violation
  ✓ test that sets up real infrastructure (DB, HTTP) to test business logic → DIP violation
""")


def main() -> None:
    print("=" * 70)
    print("SOLID PRINCIPLES — Exhaustive Notebook")
    print("=" * 70)
    notebook_srp()
    notebook_ocp()
    notebook_lsp()
    notebook_isp()
    notebook_dip()
    notebook_interlock()
    notebook_when_not_to()
    notebook_interview()
    print("\n" + "="*70)
    print("SOLID notebook complete ✔")


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()

---
## 🏆 Interview Questions — SOLID in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# SOLID in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. A route function does validation, business logic, DB access, a payment call, and email. Which SOLID principles does it violate and why does it matter?


**Deep dive.** It violates at least three. **SRP** — the function has five reasons
to change (validation rules, pricing, schema, payment API, notification), so a
change to any one risks breaking the others; changes *collide* in one place.
**OCP** — provider selection via `if/elif` means adding PayPal edits tested code.
**DIP** — it hard-codes concrete details (the DB, the gateway), so the business
logic is welded to the web framework and can only be tested by spinning up HTTP
and patching globals. Why it matters: the code becomes slow to change, risky to
modify, and nearly impossible to unit-test — the three properties that most
determine a codebase's cost over time.

---

### Q2. Explain how Dependency Inversion turns an untestable route into a fast unit test.


**Deep dive.** DIP says high-level policy depends on abstractions, not concrete
details. By extracting a `RegistrationService` that receives a `UserRepository`,
`Notifier`, and `PaymentGateway` (all Protocols) via its constructor, the service
never imports FastAPI and never constructs its own dependencies. In a test you
instantiate it with in-memory fakes and call `register()` directly — no server,
no database, milliseconds per test. In production, FastAPI's `Depends` wires the
real implementations at the composition root. The abstraction is the *seam* that
makes both substitution (swap Stripe for PayPal) and isolation (fake in tests)
possible.

---

### Q3. Where exactly should the "composition root" live in a FastAPI app, and why does it matter?


**Deep dive.** The composition root is the single place where abstractions are
bound to concrete implementations — in FastAPI, the `Depends` provider functions
(`get_service`, `get_gateway`) and the `lifespan` handler. It must sit at the
*edge* of the system so the inner layers (service, domain) stay ignorant of which
concrete DB or gateway is used. This matters because it localizes the "which
implementation?" decision: swapping in-memory for Postgres, or Stripe for a mock,
is a one-line change there and nowhere else. Scattering `new StripeGateway()`
through the code destroys that property and re-couples everything.

---

### Q4. Your teammate replaces the `if/elif` provider ladder with a dict of gateways. Is that enough to satisfy OCP?


**Deep dive.** It's a big step, but "enough" depends on the *registration*
mechanism. A dict lookup replaces the conditional, and adding a provider means
adding a class + a dict entry rather than editing branching logic — that's the
spirit of OCP. It's fully realized when new providers can be *registered* without
editing the dict's definition either (e.g., a plugin registry or entry-points), so
the core module is truly closed to modification. For most apps the dict is the
pragmatic sweet spot; over-engineering a plugin system for two providers is
YAGNI. The senior answer names the trade-off rather than claiming a single right
answer.

---

### Q5. When does applying SOLID become over-engineering in a FastAPI service?


**Deep dive.** When you introduce abstractions with a single implementation and no
test seam — e.g., a `Protocol` and a factory for a value that is stable and only
ever built one way. Every abstraction is indirection a reader must hold in their
head, and FastAPI already gives you DI cheaply, which tempts over-layering. The
heuristic: introduce an interface when there's a genuine second implementation, a
volatile external boundary (payment, email, storage), or a testing seam you
actually use. A CRUD endpoint over one table doesn't need three layers and four
Protocols. SOLID controls coupling; if there's no coupling worth controlling, the
abstraction is pure cost.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. A route function that validates, saves to DB, charges a card, and sends email violates primarily:**
- A) Liskov Substitution
- B) Single Responsibility
- C) Interface Segregation
- D) none — it's fine

**2. Selecting a payment provider with `if provider == 'stripe' elif ...` violates:**
- A) Open/Closed Principle
- B) Liskov Substitution
- C) DRY only
- D) nothing

**3. In FastAPI, the mechanism that provides Dependency Inversion is:**
- A) middleware
- B) `Depends()`
- C) `BackgroundTasks`
- D) `response_model`

**4. The main testability benefit of extracting a framework-free service layer is:**
- A) it runs faster in production
- B) business logic can be unit-tested without HTTP or a database
- C) it reduces the number of files
- D) it removes the need for Pydantic

**5. Introducing a Protocol with exactly one implementation and no test seam is usually:**
- A) required by SOLID
- B) speculative generality / over-engineering
- C) a Liskov violation
- D) necessary for OCP

### Answer Key
1. **B** — five reasons to change = SRP violation.
2. **A** — a type-switch that grows requires editing tested code (OCP).
3. **B** — `Depends()` is dependency injection built into FastAPI.
4. **B** — a framework-free service is testable in isolation.
5. **B** — abstraction without a second impl or a test seam is YAGNI.

---

## Part 3 — Gotchas Checklist

- **"God routes" fuse five concerns.** Keep routers thin: translate HTTP ⇄ domain
  and map errors to status codes; push logic to a service.
- **`if/elif` on a type code** is an OCP smell — replace with polymorphism or a
  registry/dict of strategies.
- **Business logic importing FastAPI** is the tell that it can't be unit-tested;
  the service layer must be framework-free.
- **Hidden global state** (a module-level dict, a singleton) makes tests leak into
  each other — inject dependencies instead.
- **Validate at the boundary, not everywhere.** Pydantic at the edge means inner
  layers assume clean data (don't re-validate in every function).
- **Return DTOs, not DB/domain models**, or you leak internal fields (a password
  hash) and couple your wire format to your schema.
- **Composition root at the edge only.** Constructing concretes deep in the code
  re-couples layers and defeats DIP.
- **Over-layering is also a smell.** One implementation + no test seam = delete
  the abstraction. SOLID is coupling control, not a checklist to maximize.